# 02 — Exploratory Business Analysis

This notebook is the presentation layer for validated PostgreSQL analytics. The SQL logic lives in `sql/kpi_queries.sql` and `sql/advanced_analysis.sql`; the reusable runner exports their results with:

```powershell
python scripts\run_analytics.py
```

Metric definitions and interpretation limits are documented in `docs/kpi_definitions.md`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Could not locate the project root containing src/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import REPORTS_DIR

ANALYTICS_DIR = REPORTS_DIR / 'analytics'
pio.renderers.default = 'notebook_connected'
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

## Load validated analytical extracts

In [2]:
extract_names = [
    'executive_summary', 'monthly_performance', 'category_performance',
    'state_performance', 'payment_method_performance',
    'order_status_distribution', 'delivery_review_relationship',
    'customer_purchase_frequency', 'monthly_customer_cohorts',
    'seller_scorecard',
]
missing = [
    ANALYTICS_DIR / f'{name}.csv'
    for name in extract_names
    if not (ANALYTICS_DIR / f'{name}.csv').is_file()
]
if missing:
    raise FileNotFoundError(
        'Analytics extracts are missing. From the repository root run: '
        'python scripts\\run_analytics.py'
    )

extracts = {
    name: pd.read_csv(ANALYTICS_DIR / f'{name}.csv')
    for name in extract_names
}
monthly = extracts['monthly_performance']
monthly['order_month'] = pd.to_datetime(monthly['order_month'])
cohorts = extracts['monthly_customer_cohorts']
cohorts['cohort_month'] = pd.to_datetime(cohorts['cohort_month'])
print(f'Loaded {len(extracts)} validated analytical extracts from reports/analytics.')

Loaded 10 validated analytical extracts from reports/analytics.


## Executive KPI snapshot

Delivered GMV is item price only and excludes freight. Repeat customers use `customer_unique_id` and at least two delivered orders.

In [3]:
executive = extracts['executive_summary'].iloc[0]
executive_display = pd.DataFrame(
    {
        'Metric': [
            'Observed period', 'Total orders', 'Delivered orders',
            'Delivered GMV', 'Delivered AOV', 'Cancellation rate',
            'On-time delivery rate', 'Average delivery days',
            'Average review score', 'Low-review rate',
            'Repeat-customer rate', 'Active sellers',
        ],
        'Value': [
            f"{executive['first_order_date']} to {executive['last_order_date']}",
            f"{executive['total_orders']:,.0f}",
            f"{executive['delivered_orders']:,.0f}",
            f"R${executive['delivered_gmv']:,.2f}",
            f"R${executive['delivered_average_order_value']:,.2f}",
            f"{executive['cancellation_rate_pct']:.2f}%",
            f"{executive['on_time_delivery_rate_pct']:.2f}%",
            f"{executive['average_delivery_days']:.2f}",
            f"{executive['average_review_score']:.2f}",
            f"{executive['low_review_rate_pct']:.2f}%",
            f"{executive['repeat_customer_rate_pct']:.2f}%",
            f"{executive['active_sellers']:,.0f}",
        ],
    }
)
display(executive_display)

,Metric,Value
0,Observed period,2016-09-04 to 2018-10-17
1,Total orders,"99,441"
2,Delivered orders,"96,478"
3,Delivered GMV,"R$13,221,498.11"
4,Delivered AOV,R$137.04
5,Cancellation rate,0.63%
6,On-time delivery rate,93.23%
7,Average delivery days,12.56
8,Average review score,4.09
9,Low-review rate,14.64%


## Monthly commercial and operational trend

The chart uses January 2017 through August 2018. Boundary months outside this window are incomplete and remain available in the extract.

In [4]:
core_monthly = monthly.loc[
    monthly['order_month'].between('2017-01-01', '2018-08-01')
].copy()
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(
    go.Bar(
        x=core_monthly['order_month'], y=core_monthly['delivered_gmv'],
        name='Delivered GMV', hovertemplate='%{x|%b %Y}<br>R$%{y:,.0f}<extra></extra>',
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=core_monthly['order_month'], y=core_monthly['on_time_delivery_rate_pct'],
        name='On-time delivery', mode='lines+markers',
        hovertemplate='%{x|%b %Y}<br>%{y:.1f}%<extra></extra>',
    ),
    secondary_y=True,
)
fig.update_yaxes(title_text='Delivered GMV (R$)', secondary_y=False)
fig.update_yaxes(title_text='On-time delivery (%)', range=[70, 100], secondary_y=True)
fig.update_layout(title='Monthly delivered GMV and on-time performance', hovermode='x unified')
fig.show()

## Category contribution

In [5]:
top_categories = extracts['category_performance'].head(10).sort_values('delivered_gmv')
fig = px.bar(
    top_categories, x='delivered_gmv', y='category_name', orientation='h',
    labels={'delivered_gmv': 'Delivered GMV (R$)', 'category_name': 'Category'},
    title='Top 10 categories by delivered GMV',
    hover_data=['delivered_orders', 'items_sold', 'average_review_score'],
)
fig.update_traces(texttemplate='R$%{x:,.0f}', textposition='outside')
fig.show()

## Delivery timing and customer experience

This is an observed association. It does not prove that delivery timing alone caused each review.

In [6]:
delivery = extracts['delivery_review_relationship'].sort_values('band_order')
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(
    go.Bar(
        x=delivery['delivery_timing_band'], y=delivery['low_review_rate_pct'],
        name='Low-review rate', text=delivery['low_review_rate_pct'],
        texttemplate='%{text:.1f}%', hovertemplate='%{x}<br>%{y:.1f}% low reviews<extra></extra>',
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=delivery['delivery_timing_band'], y=delivery['average_review_score'],
        name='Average review', mode='lines+markers+text',
        text=delivery['average_review_score'], textposition='top center',
        hovertemplate='%{x}<br>%{y:.2f} average review<extra></extra>',
    ),
    secondary_y=True,
)
fig.update_yaxes(title_text='Low-review rate (%)', range=[0, 100], secondary_y=False)
fig.update_yaxes(title_text='Average review score', range=[1, 5], secondary_y=True)
fig.update_layout(title='Review outcomes deteriorate as delivery delays increase')
fig.show()

## Repeat purchasing

In [7]:
frequency = extracts['customer_purchase_frequency'].sort_values('band_order')
fig = px.bar(
    frequency, x='frequency_band', y='customer_share_pct',
    labels={'frequency_band': 'Delivered-order frequency', 'customer_share_pct': 'Customers (%)'},
    title='Customer distribution by delivered-order frequency',
    text='customer_share_pct', hover_data=['customers'],
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_yaxes(range=[0, 105])
fig.show()

## Delivered-order cohorts

Recent cohorts have fewer observable follow-up months, so compare cohorts only at common month numbers.

In [8]:
cohort_matrix = cohorts.pivot(
    index='cohort_month', columns='month_number', values='retention_rate_pct'
).sort_index()
cohort_matrix.index = cohort_matrix.index.strftime('%Y-%m')
fig = px.imshow(
    cohort_matrix, aspect='auto', text_auto='.2f',
    labels={'x': 'Months since first delivered order', 'y': 'Cohort month', 'color': 'Retention %'},
    title='Monthly delivered-order cohort activity (%)',
)
fig.show()

## Geographic concentration

In [9]:
top_states = extracts['state_performance'].head(10).sort_values('delivered_gmv')
fig = px.bar(
    top_states, x='delivered_gmv', y='customer_state', orientation='h',
    labels={'delivered_gmv': 'Delivered GMV (R$)', 'customer_state': 'Customer state'},
    title='Top 10 customer states by delivered GMV',
    hover_data=['delivered_orders', 'on_time_delivery_rate_pct', 'average_review_score'],
)
fig.update_traces(texttemplate='R$%{x:,.0f}', textposition='outside')
fig.show()

## Calculated takeaways

In [10]:
categories = extracts['category_performance']
states = extracts['state_performance']
peak = core_monthly.loc[core_monthly['delivered_gmv'].idxmax()]
worst_on_time = core_monthly.loc[core_monthly['on_time_delivery_rate_pct'].idxmin()]
top_five_category_share = 100 * categories.head(5)['delivered_gmv'].sum() / executive['delivered_gmv']
top_three_state_share = 100 * states.head(3)['delivered_gmv'].sum() / executive['delivered_gmv']
late = delivery.loc[delivery['delivery_timing_band'].eq('8+ days late')].iloc[0]

print(f"Peak core-window GMV: {peak['order_month']:%B %Y}, R${peak['delivered_gmv']:,.2f}.")
print(
    f"Lowest core-window on-time rate: {worst_on_time['order_month']:%B %Y}, "
    f"{worst_on_time['on_time_delivery_rate_pct']:.2f}%."
)
print(f"Top five categories: {top_five_category_share:.2f}% of delivered GMV.")
print(f"Top three customer states: {top_three_state_share:.2f}% of delivered GMV.")
print(
    f"Orders 8+ days late: {late['average_review_score']:.2f} average review and "
    f"{late['low_review_rate_pct']:.2f}% low-review rate."
)

Peak core-window GMV: November 2017, R$987,765.37.
Lowest core-window on-time rate: March 2018, 81.04%.
Top five categories: 39.83% of delivered GMV.
Top three customer states: 63.38% of delivered GMV.
Orders 8+ days late: 1.70 average review and 79.18% low-review rate.


## Business communication

The evidence-based narrative and recommendations are maintained in `docs/analysis_findings.md`. The next phase should convert the most decision-relevant metrics into an interactive stakeholder dashboard rather than adding more undirected charts.